# Mental Health and Academic Performance Predictor

This notebook analyzes how college students' mental health, discipline, sleep, and screen-time habits relate to GPA. It walks through:

1. **Data loading & cleaning** (`DataPipeline`)
2. **Dataset summary** (`data_summary`)
3. **Exploratory Data Analysis** (`data_eda`)
4. **Model training & evaluation** (`ModelTrainer`)
5. **Interactive GPA prediction**

> **Setup:** Download the dataset from Kaggle, rename the CSV to `students.csv`, and place it in the `data/` folder before running.
>
> Dataset URL: https://www.kaggle.com/datasets/sharmajicoder/college-students-habits-and-performance

## 0. Imports & Setup

In [ ]:
# Standard library
import math
import itertools
from functools import reduce

# Third-party
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Project modules
from data_pipeline import DataPipeline, FEATURE_COLUMNS, TARGET_COLUMN
from data_summary import print_data_summary
from data_eda import run_eda
from model_trainer import ModelTrainer
from utils import find_missing_columns

%matplotlib inline
plt.rcParams["figure.dpi"] = 110
print("All modules imported successfully.")

## 1. Load & Clean the Data

In [ ]:
# Instantiate the pipeline — one of the two required classes.
# DataPipeline composes with ModelTrainer (composition relationship).
pipeline = DataPipeline(csv_path="data/students.csv")

# Exception-handling scenario 1: missing CSV file
try:
    df = pipeline.clean()
except FileNotFoundError as e:
    print(f"[ERROR] {e}")
    raise

print(f"Loaded {len(df):,} rows x {df.shape[1]} columns after cleaning.")
df.head(3)

## 2. Dataset Summary

In [ ]:
print_data_summary()

## 3. Feature & Target Split

In [ ]:
# Exception-handling scenario 2: schema mismatch (missing required columns)
try:
    X, y = pipeline.get_features_and_target()
except ValueError as e:
    print(f"[ERROR] {e}")
    raise

print(f"Features : {X.shape[1]} columns")
print(f"Target   : '{y.name}' — range [{y.min():.3f}, {y.max():.3f}]")

## 4. Exploratory Data Analysis

In [ ]:
# Calls run_eda() from data_eda.py — prints correlations and shows both charts
run_eda(X, y)

In [ ]:
# List comprehension (Part 2 requirement)
df_num = pd.concat([X, y], axis=1).select_dtypes(include="number")
correlations = df_num.corr()["gpa"].drop("gpa").sort_values(key=abs, ascending=False)

strong_predictors = [feat for feat, r in correlations.items() if abs(r) > 0.3]
print("Features with |r| > 0.3:", strong_predictors)

In [ ]:
# Generator function (Part 2 requirement)
def feature_stats_generator(dataframe):
    """Yield (feature_name, mean, std) tuples one at a time."""
    for col in dataframe.columns:
        yield col, dataframe[col].mean(), dataframe[col].std()

print(f"{'Feature':<35} {'Mean':>8} {'Std':>8}")
print("-" * 55)
for name, mean, std in feature_stats_generator(X):
    print(f"{name:<35} {mean:>8.3f} {std:>8.3f}")

In [ ]:
# Set operations (Part 2 requirement)
# Mutable type: set; immutable types: strings inside the set
all_features: set = set(X.columns)
high_correlation_features: set = set(strong_predictors)
low_correlation_features = all_features - high_correlation_features

print("Total features    :", len(all_features))
print("High |r| (> 0.3)  :", high_correlation_features)
print("Low  |r| (<= 0.3) :", low_correlation_features)

In [ ]:
# reduce() and lambda (Part 2 requirement)
# Cumulative sum of individual R² values
r_squared_values: list = [r ** 2 for r in correlations.values]  # list (mutable)
cumulative_r2 = reduce(lambda acc, v: acc + v, r_squared_values, 0.0)

# Tuple (immutable)
summary_tuple: tuple = ("cumulative_r2", round(cumulative_r2, 4))
print(f"Sum of individual R² values: {summary_tuple}")

## 5. Model Training

In [ ]:
# ModelTrainer is the second required class.
# It receives a DataPipeline — this is the composition relationship.
trainer = ModelTrainer(pipeline)

# __str__ and __len__ operator overload demos
print(trainer)
print(f"Training rows: {len(trainer)}")

model = trainer.train()
print("\nModel trained successfully.")

## 6. Model Evaluation

In [ ]:
metrics = trainer.evaluate()

In [ ]:
# Residual plots
predictions = trainer.predict(trainer.X_test)
residuals = trainer.y_test - predictions

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Predicted vs actual
axes[0].scatter(trainer.y_test, predictions, alpha=0.4, color="steelblue", edgecolors="none")
lims = [min(trainer.y_test.min(), predictions.min()),
        max(trainer.y_test.max(), predictions.max())]
axes[0].plot(lims, lims, "--", color="red", linewidth=1)
axes[0].set_xlabel("Actual GPA")
axes[0].set_ylabel("Predicted GPA")
axes[0].set_title("Actual vs Predicted GPA")

# Residual distribution
axes[1].hist(residuals, bins=30, edgecolor="black", color="steelblue")
axes[1].axvline(0, color="red", linestyle="--", linewidth=1)
axes[1].set_xlabel("Residual (Actual - Predicted)")
axes[1].set_ylabel("Count")
axes[1].set_title("Residual Distribution")

plt.tight_layout()
plt.show()

In [ ]:
# enumerate() and zip() (Part 2 requirement)
coef_pairs = list(zip(FEATURE_COLUMNS, model.coef_))
coef_pairs.sort(key=lambda x: abs(x[1]), reverse=True)

print("Top 5 most influential features (by |coefficient|):")
for rank, (feat, coef) in enumerate(coef_pairs[:5], start=1):
    direction = "up GPA" if coef > 0 else "down GPA"
    print(f"  {rank}. {feat:<35} coef = {coef:+.4f}  ({direction})")

In [ ]:
# map() (Part 2 requirement)
top3 = [feat for feat, _ in coef_pairs[:3]]
top3_upper = list(map(lambda c: c.upper(), top3))
print("Top 3 features (uppercased via map):", top3_upper)

In [ ]:
# itertools built-in library (Part 2 requirement)
print("All pairs from top-3 features (itertools.combinations):")
for pair in itertools.combinations(top3, 2):
    print(" ", pair)

## 7. Predict GPA for a Custom Student

Edit the values below and re-run to get an instant GPA prediction.

In [ ]:
# Dict (mutable) with number and string values (immutable)
student_profile: dict = {
    "stress"                : 5,
    "anxiety"               : 4,
    "depression"            : 3,
    "motivation"            : 7,
    "concentration"         : 7,
    "time_management"       : 7,
    "self_discipline"       : 8,
    "procrastination_score" : 3,
    "financial_stress"      : 4,
    "sleep_hours"           : 7,
    "late_night_frequency"  : 2,
    "screen_time"           : 4,
    "phone_unlocks_per_day" : 50,
    "previous_gpa"          : 7.5,
}

# Exception-handling scenario 3: invalid input values
try:
    if not (0 <= student_profile["sleep_hours"] <= 24):
        raise ValueError("sleep_hours must be between 0 and 24.")
    if not (0 <= student_profile["previous_gpa"] <= 10):
        raise ValueError("previous_gpa must be between 0 and 10.")
except ValueError as e:
    print(f"[INPUT ERROR] {e}")
    raise

student_df = pd.DataFrame([student_profile])[FEATURE_COLUMNS]
predicted_gpa = trainer.predict(student_df).iloc[0]

print(f"Predicted GPA : {predicted_gpa:.3f}")
print(f"Dataset range : {y.min():.3f} - {y.max():.3f}")

## 8. Run Tests

In [ ]:
import subprocess
result = subprocess.run(
    ["pytest", "-v", "test_data_pipeline.py", "test_model_trainer.py"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

## 9. `__name__` Guard Demo (Part 2)

All `.py` modules use `if __name__ == "__main__":` so they can be imported without side effects and also run standalone. The same pattern is demonstrated here:

In [ ]:
if __name__ == "__main__":
    print("Running as main — mirrors the guard used in all .py modules.")